<p align="center">
  <strong>Run this notebook:</strong>
</p>

<p align="center">
  <a href="https://colab.research.google.com/github/MeteoSwiss/nwp-fdb-polytope-demo/blob/main/examples/earthkit/earthkit_onboarding_meteoswiss.ipynb">
    <img
      src="https://colab.research.google.com/assets/colab-badge.svg"
      alt="Open the main version in Google Colab"
    >
  </a>
  &nbsp;
  <a href="https://colab.research.google.com/github/MeteoSwiss/nwp-fdb-polytope-demo/blob/intro_earthkit_notebook/examples/earthkit/earthkit_onboarding_meteoswiss.ipynb">
    <img
      src="https://img.shields.io/badge/Open_dev_version-in%20Colab-F9AB00?logo=googlecolab&logoColor=white"
      alt="Open the development version in Google Colab"
    >
  </a>
</p>

<br>

<p align="center">
  <a href="https://earthkit.ecmwf.int/">
    <img
      src="https://github.com/ecmwf/logos/raw/refs/heads/main/logos/earthkit/earthkit-light.svg"
      alt="earthkit"
      width="520"
    >
  </a>
</p>

<h1 align="center">earthkit onboarding for MeteoSwiss NWP workflows</h1>

> **What is earthkit?**
>
> earthkit is an open-source Python ecosystem led by ECMWF. It provides a
> consistent workflow for accessing, inspecting, processing, analysing, and
> visualising weather and climate data—including the GRIB data commonly used
> in numerical weather prediction.

<p align="center">
  <a href="https://earthkit.ecmwf.int/">Website</a>
  &nbsp;·&nbsp;
  <a href="https://earthkit.readthedocs.io/en/latest/">Documentation</a>
  &nbsp;·&nbsp;
  <a href="https://github.com/ecmwf/earthkit">GitHub</a>
</p>

---

<p align="center">
  <img
    src="https://raw.githubusercontent.com/MeteoSwiss/nwp-fdb-polytope-demo/intro_earthkit_notebook/examples/earthkit/earthkit_components.png"
    alt="Overview of earthkit components"
    width="720"
  >
</p>

## earthkit components

| Package | Purpose |
|---|---|
| [`earthkit-data`](https://earthkit-data.readthedocs.io/en/stable/) | A format-agnostic Python interface for geospatial data, with a focus on meteorology and climate science. |
| [`earthkit-plots`](https://earthkit-plots.readthedocs.io/en/stable/) | Produce publication-quality weather and climate charts and maps with only a few lines of code. |
| [`earthkit-meteo`](https://earthkit-meteo.readthedocs.io/en/stable/) | Perform common meteorological calculations using NumPy, Torch, CuPy, xarray, or field lists. |
| [`earthkit-geo`](https://earthkit-geo.readthedocs.io/en/stable/) | Work with geospatial shapes, coordinates, grids, and projections. |
| [`earthkit-transforms`](https://earthkit-transforms.readthedocs.io/en/stable/) | Apply transformations, aggregations, and statistical analyses across data cubes. |
| [`earthkit-hydro`](https://earthkit-hydro.readthedocs.io/en/stable/) | Work with river networks, flow accumulation, and other hydrological data. |

---

## Why earthkit at MeteoSwiss?

MeteoSwiss uses gridded meteorological data across forecasting, research and machine-learning workflows. Many of these workflows repeat similar tasks, such as reading data, inspecting metadata, converting formats, regridding and plotting.

Using shared earthkit components can help to reduce duplicated MeteoSwiss-specific implementations and align with tools used across the meteorological community.

The release of earthkit 1.0 on 2 July 2026 marked its core interfaces as stable for wider research and operational use.

![earthkit-data-logo](https://github.com/ecmwf/logos/raw/refs/heads/main/logos/earthkit/earthkit-data-light.svg)

## The mental model to remember

The most useful earthkit mental model is:

```text
source → earthkit data object → FieldList / Xarray / NumPy / Pandas → analysis or plotting
```

## Environment setup

In [1]:
# 📦 Notebook setup: Colab
import sys, os, pathlib

IN_COLAB = "google.colab" in sys.modules

# TODO rm -b update-deps when PR is merged to main
#if IN_COLAB:
!git clone -b update-deps https://github.com/MeteoSwiss/nwp-fdb-polytope-demo.git
%cd nwp-fdb-polytope-demo
!pip install poetry && poetry config virtualenvs.in-project true && poetry install --no-ansi

fatal: destination path 'nwp-fdb-polytope-demo' already exists and is not an empty directory.
/scratch/mch/nburgdor/02_iwf/intro_earthkit/nwp-fdb-polytope-demo/examples/earthkit/nwp-fdb-polytope-demo
Looking in indexes: https://service.meteoswiss.ch/nexus/repository/python-all/simple
Skipping virtualenv creation, as specified in config file.
Installing dependencies from lock file

No dependencies to install or update


In [2]:
import earthkit.data as ekd

print("earthkit-data:", ekd.__version__)

earthkit-data: 1.0.2


Read the data. Here source == file but could be url, FDB, Polytope... All these sources: https://earthkit-data.readthedocs.io/en/latest/concepts/inputs/from_source.html#from_source

In [3]:
import earthkit.data as ekd

GRIB_FILE = "icon-ch2-eps-202607131200-100-t_2m-ctrl.grib2"

data = ekd.from_source("file", GRIB_FILE)
data

FileNotFoundError: No such file exists: 'icon-ch2-eps-202607131200-100-t_2m-ctrl.grib2'

In [ ]:
print("Available conversions:", data.available_types)

Available conversions: ['fieldlist', 'pandas', 'xarray', 'numpy', 'array']


In [ ]:
fl = data.to_fieldlist()
print("Number of fields/messages:", len(fl))
fl.ls()

Number of fields/messages: 1


,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,2t,2026-07-17 16:00:00,2026-07-13 12:00:00,4 days 04:00:00,2,height_above_ground_level,0,ICON


The object returned by `from_source()` is the entry point. It can expose several representations of the same source data. For GRIB, the most useful first representation is a `FieldList`.


### Checkpoint

Look at the table above before continuing:

- How many GRIB messages are present?
- Do the parameters, times, levels and grid types look plausible?
- Are any parameters shown as `unknown`?

`unknown` or suspicious values are a reason to investigate ecCodes definitions before assuming that earthkit is decoding incorrectly.


## 7. Inspect one field


In [ ]:
field = fl[0]
field

number_of_values,283876
array_type,ndarray
array_dtype,float64
variable,2t
standard_name,air_temperature
long_name,2 metre temperature
units,kelvin
valid_datetime,2026-07-17 16:00:00
base_datetime,2026-07-13 12:00:00
step,"4 days, 4:00:00"
level,2


In [ ]:
values = field.to_numpy()

print("Array shape:", values.shape)
print("Field shape:", field.shape)
print("dtype:      ", values.dtype)
print("minimum:    ", float(values.min()))
print("mean:       ", float(values.mean()))
print("maximum:    ", float(values.max()))

Array shape: (283876,)
Field shape: (283876,)
dtype:       float64
minimum:     272.8619079589844
mean:        299.98226170856384
maximum:     314.6187438964844


### earthkit 1.0 metadata

Prefer format-independent metadata keys for normal application logic. They make code less dependent on GRIB-specific key names and help the same workflow work across formats.


In [ ]:
high_level_keys = [
    "parameter.variable",
    "parameter.name",
    "parameter.units",
    "time.base_datetime",
    "time.step",
    "time.valid_datetime",
    "vertical.level",
    "vertical.level_type",
    "ensemble.member",
    "geography.grid_type",
]

for key in high_level_keys:
    try:
        value = field.get(key)
    except Exception as exc:
        value = f"<error: {type(exc).__name__}: {exc}>"
    print(f"{key:28s} {value}")

parameter.variable           2t
parameter.name               None
parameter.units              kelvin
time.base_datetime           2026-07-13 12:00:00
time.step                    4 days, 4:00:00
time.valid_datetime          2026-07-17 16:00:00
vertical.level               2
vertical.level_type          height_above_ground_level
ensemble.member              0
geography.grid_type          ICON


### Raw GRIB / ecCodes metadata

Use raw metadata when debugging GRIB-specific behaviour, definitions or encoding. With earthkit 1.0, raw keys can be accessed through `metadata()` or with the `metadata.` prefix in `get()`.


In [ ]:
raw_keys = [
    "edition",
    "centre",
    "shortName",
    "name",
    "units",
    "gridType",
    "typeOfLevel",
    "level",
    "numberOfDataPoints",
    "Ni",
    "Nj",
]

for key in raw_keys:
    try:
        value = field.metadata(key)
    except Exception:
        value = None
    print(f"{key:22s} {value}")

print("\nEquivalent prefixed access examples:")
print("metadata.shortName:", field.get("metadata.shortName"))
print("metadata.gridType: ", field.get("metadata.gridType"))

edition                2
centre                 lssw
shortName              2t
name                   2 metre temperature
units                  K
gridType               unstructured_grid
typeOfLevel            heightAboveGround
level                  2
numberOfDataPoints     283876
Ni                     None
Nj                     None

Equivalent prefixed access examples:
metadata.shortName: 2t
metadata.gridType:  unstructured_grid


### Exercise 1 — answer from metadata

Fill these values without inspecting the raw file manually:

- parameter:
- units:
- valid time:
- vertical level and level type:
- grid type:

Then compare the high-level and raw metadata. Which form would you use in production application code, and which form would you use while debugging a GRIB definition problem?


## 8. FieldList operations


In [ ]:
# Values for a key across all fields.
print("Parameters:", fl.get("parameter.variable"))
print("Valid times:", fl.get("time.valid_datetime"))
print("Grid types: ", fl.get("geography.grid_type"))


Parameters: ['2t']
Valid times: [datetime.datetime(2026, 7, 17, 16, 0)]
Grid types:  ['ICON']


In [ ]:
# Select fields by format-independent metadata.
first_parameter = field.get("parameter.variable")
selected = fl.sel({"parameter.variable": first_parameter})

print(f"Selected {len(selected)} field(s) for parameter {first_parameter!r}")
selected.ls()


FieldList is the right level for message-oriented operations: listing, selecting, iterating over fields, comparing metadata, or writing subsets. Avoid assuming that `fl[0]` is always the field you want in production code—select explicitly by parameter, time, level and member.


## 9. Convert to Xarray


In [ ]:
ds = data.to_xarray()
ds

FileNotFoundError: [Errno 2] No such file or directory: 'icon-ch2-eps-202607131200-100-t_2m-ctrl.grib2'

In earthkit-data 1.0, the default Xarray profile is named `earthkit`. It uses format-independent field metadata to create dimensions and coordinates.

Inspect the structure before analysis. ICON native-grid data may use a point-like horizontal dimension rather than `latitude × longitude`.


In [ ]:
print("Dimensions:", dict(ds.sizes))
print("Coordinates:", list(ds.coords))
print("Data variables:", list(ds.data_vars))

first_variable_name = next(iter(ds.data_vars))
da = ds[first_variable_name]
print("\nFirst DataArray:", first_variable_name)
da


In [ ]:
# A tiny, grid-agnostic analysis example.
summary = {
    "min": da.min(skipna=True).item(),
    "mean": da.mean(skipna=True).item(),
    "max": da.max(skipna=True).item(),
}
summary


### Exercise 2 — explore the Xarray representation

1. Which dimensions describe forecast time, level, ensemble member and horizontal space?
2. Is the valid time a dimension, coordinate or attribute?
3. Is the grid represented as two horizontal dimensions or a single point/cell dimension?
4. Select one variable and, where relevant, one time, level and member.

Do not copy selection code blindly between regular latitude/longitude and ICON native-grid datasets. First inspect `ds.dims`, `ds.coords` and the variable names.


## 10. NumPy and Pandas views


In [ ]:
# NumPy is useful for numerical kernels, but metadata is no longer the main interface.
arr = field.to_numpy()
print("NumPy shape:", arr.shape)

# For large GRIB collections, converting everything to Pandas may be expensive.
# Here we only demonstrate the interface on the tiny sample.
try:
    frame = data.to_pandas()
    print("Pandas shape:", frame.shape)
    display(frame.head())
except Exception as exc:
    print("Pandas conversion is not available for this source/configuration:", exc)


Use NumPy when a numerical algorithm specifically needs arrays. Keep the FieldList or Xarray object nearby so metadata and coordinate meaning are not lost.


## 11. Quick plotting


In [ ]:
if ekp is None:
    print("Install earthkit-plots to run this section.")
else:
    # quickplot infers a suitable representation and style from the data and metadata.
    ekp.quickplot(field)


`earthkit.plots.quickplot()` is the preferred high-level entry point in the 1.0 API. It can plot many regular and non-regular grids, but successful direct plotting still depends on decoded coordinates and supported grid metadata.

A plot that renders is not automatically scientifically valid. Check:

- parameter and units;
- valid time and level;
- grid type and coordinate interpretation;
- whether interpolation or regridding has occurred;
- whether the visualisation method is appropriate for cells or points.

See [earthkit-plots documentation](https://earthkit-plots.readthedocs.io/en/latest/).


## 12. MeteoSwiss-specific topic: ecCodes definitions

A GRIB message contains numeric codes and compact metadata, not always a complete human-readable description. ecCodes definition files map those codes to names, units, level types and other attributes.

For standard GRIB data, the definitions shipped with ecCodes are usually sufficient. ICON/COSMO-related files can require additional COSMO definitions. Missing or mismatched definitions can lead to symptoms such as:

- `shortName = unknown`;
- missing or surprising units;
- unexpected level types;
- several messages that should describe different parameters appearing identical.

**Important:** definitions must be configured **before** the GRIB is opened.

Useful references:

- [MeteoSwiss Open Data: setting COSMO definitions](https://opendatadocs.meteoswiss.ch/e-forecast-data/e2-e3-numerical-weather-forecasting-model)
- [COSMO-ORG/eccodes-cosmo-resources](https://github.com/COSMO-ORG/eccodes-cosmo-resources)
- [MeteoSwiss/eccodes-cosmo-resources-python](https://github.com/MeteoSwiss/eccodes-cosmo-resources-python)
- [Internal: GRIB and NetCDF file handling tips & tricks](https://meteoswiss.atlassian.net/wiki/spaces/APN/pages/1901816/GRIB+and+NetCDF+File+handling+tipps+tricks)


In [ ]:
# Inspect definition-related environment variables visible to this kernel.
for variable in ("ECCODES_DEFINITION_PATH", "GRIB_DEFINITION_PATH", "ECCODES_SAMPLES_PATH"):
    print(f"{variable:26s} {os.environ.get(variable, '<not set>')}")


A typical shell setup places the COSMO definitions before the vendor definitions:

```bash
export ECCODES_DEFINITION_PATH=/path/to/eccodes-cosmo-resources/definitions:/path/to/eccodes/definitions
```

Some existing COSMO documentation and host programs refer to `GRIB_DEFINITION_PATH`. Check `codes_info` and follow the supported convention in the MeteoSwiss environment you are using.

A Python-based setup is also possible with `eccodes-cosmo-resources-python`:

```python
import eccodes
import eccodes_cosmo_resources

vendor = eccodes.codes_definition_path()
cosmo = eccodes_cosmo_resources.get_definitions_path()
eccodes.codes_set_definitions_path(f"{cosmo}:{vendor}")
```

Run this before importing/reading the target GRIB in a clean kernel.


### Run ecCodes command-line diagnostics


In [ ]:
def run_command(args: list[str], max_lines: int = 30) -> None:
    executable = shutil.which(args[0])
    if executable is None:
        print(f"{args[0]!r} is not available on PATH")
        return

    result = subprocess.run(
        [executable, *args[1:]],
        check=False,
        capture_output=True,
        text=True,
    )
    output = (result.stdout or "") + (result.stderr or "")
    lines = output.splitlines()
    print("\n".join(lines[:max_lines]))
    if len(lines) > max_lines:
        print(f"... ({len(lines) - max_lines} more line(s))")
    print("return code:", result.returncode)

run_command(["codes_info"])


In [ ]:
run_command([
    "grib_ls",
    "-p",
    "edition,centre,date,dataType,gridType,stepRange,typeOfLevel,level,shortName,units,packingType",
    str(grib_path),
])


In [ ]:
# The dump is intentionally truncated. Remove max_lines only when you really need the full output.
run_command(["grib_dump", "-O", str(grib_path)], max_lines=25)


### Diagnostic habit

When earthkit metadata looks suspicious:

1. compare high-level and raw metadata;
2. run `codes_info` to confirm versions and definition paths;
3. run `grib_ls` on the same file;
4. confirm the COSMO definition release matches the ecCodes vendor version;
5. restart the kernel after changing definition paths;
6. reopen the GRIB and compare again.


## 13. MeteoSwiss-specific topic: ICON grids

ICON data is commonly stored on a non-regular, unstructured grid. Do not assume that:

- values can be reshaped to `(latitude, longitude)`;
- neighbouring array elements are neighbouring geographic cells;
- a generic `imshow()` is meaningful;
- a regular-grid interpolation method is valid.

Always inspect the grid first.


In [ ]:
grid_checks = {
    "earthkit grid type": field.get("geography.grid_type"),
    "raw gridType": field.get("metadata.gridType"),
    "number of data points": field.get("metadata.numberOfDataPoints"),
    "Ni": field.get("metadata.Ni"),
    "Nj": field.get("metadata.Nj"),
}

grid_checks


In [ ]:
grid_type = field.get("geography.grid_type")

if grid_type in {"regular_ll", "regular_gg"}:
    print("This is a regular grid. Direct plotting and Xarray operations are usually straightforward.")
elif grid_type in {"unstructured_grid", "unstructured"}:
    print(
        "This is an unstructured grid. earthkit-plots may plot it directly when coordinates "
        "and grid metadata are available, but do not reshape values into a regular 2-D grid."
    )
else:
    print(
        f"Grid type {grid_type!r} needs inspection. Check earthkit-plots support and whether "
        "coordinates/definitions are complete before plotting or regridding."
    )


For an ICON file, also inspect whether latitude and longitude coordinates are available and whether they represent cell centres, vertices or another topology. The correct interpretation matters for plotting, spatial subsets and conservative regridding.


## 14. Regridding: keep it explicit and validated

Regridding is a separate processing decision, not a cosmetic plotting fix.

Before regridding, document:

- source grid and resolution;
- target grid and resolution;
- interpolation/remapping method;
- treatment of missing data and boundaries;
- whether the quantity is intensive or extensive;
- validation metrics and tolerances;
- provenance in the output metadata.

For the first onboarding, the important capability is to identify the current grid and explain **why** regridding may be needed. Do not silently regrid merely to obtain a rectangular array.

MeteoSwiss branch example: [work with data on the native grid / regridding](https://github.com/MeteoSwiss/nwp-fdb-polytope-demo/blob/native_grid-16022026/examples/Polytope/work_with_data_native_grid.ipynb)


## 15. Capstone: produce a one-field onboarding report

The helper below creates a compact, reusable report for the first field. Run it on the sample and then on a MeteoSwiss GRIB.


In [ ]:
def first_field_report(field) -> dict[str, object]:
    'Return the metadata needed by the onboarding checklist.'
    return {
        "parameter": field.get("parameter.variable"),
        "parameter_name": field.get("parameter.name"),
        "units": field.get("parameter.units"),
        "valid_time": field.get("time.valid_datetime"),
        "vertical_level": field.get("vertical.level"),
        "vertical_level_type": field.get("vertical.level_type"),
        "grid_type": field.get("geography.grid_type"),
        "number_of_data_points": field.get("metadata.numberOfDataPoints"),
        "raw_short_name": field.get("metadata.shortName"),
        "raw_grid_type": field.get("metadata.gridType"),
        "shape": field.shape,
    }

report = first_field_report(field)
report


In [ ]:
def plotting_assessment(report: dict[str, object]) -> str:
    grid_type = report["grid_type"]
    parameter = report["parameter"]
    units = report["units"]

    if not parameter or parameter == "unknown" or not units:
        return (
            "Do not trust the plot yet: parameter metadata is incomplete. "
            "Check ecCodes/COSMO definitions first."
        )
    if grid_type in {"regular_ll", "regular_gg"}:
        return "Likely suitable for direct earthkit quick plotting; still verify coordinates and units."
    if grid_type in {"unstructured_grid", "unstructured"}:
        return (
            "Potentially suitable for direct earthkit plotting when coordinates/topology are decoded. "
            "Do not reshape; validate the native-grid representation."
        )
    return "Needs grid-specific investigation before plotting or regridding."

print(plotting_assessment(report))


### Final participant checklist

You are done when you can answer, in your own words:

- **Which parameter is in the file?**
- **What is the valid time?**
- **What vertical level is used?**
- **Which grid type is used?**
- **Can it be plotted directly? Why or why not?**
- **When would you inspect raw GRIB metadata rather than earthkit’s high-level metadata?**
- **What would make you suspect a missing or mismatched ecCodes definition set?**
- **Why must regridding be treated as an explicit, validated processing step?**


## 16. Further learning and examples

### Core documentation

- [earthkit](https://earthkit.readthedocs.io/en/latest/)
- [earthkit-data](https://earthkit-data.readthedocs.io/en/latest/)
- [earthkit-plots](https://earthkit-plots.readthedocs.io/en/latest/)
- [earthkit-geo](https://earthkit-geo.readthedocs.io/en/latest/)
- [earthkit-data 1.0 migration guide](https://earthkit-data.readthedocs.io/en/latest/release-notes/migration_1.0.0.html)

### Training and notebooks

- [ECMWF 2026 earthkit training](https://github.com/ecmwf-training/2026-earthkit-training/tree/main)
- [MeteoSwiss introductory earthkit branch notebook](https://github.com/MeteoSwiss/opendata-nwp-demos/blob/intro_earthkit/00_intro_earthkit_2.ipynb)
- [MeteoSwiss ICON-CH2 pollen forecast](https://github.com/MeteoSwiss/opendata-nwp-demos/blob/main/10_icon_ch2_pollen_forecast.ipynb)
- [MeteoSwiss Polytope polygon cut-out example](https://htmlpreview.github.io/?https://raw.githubusercontent.com/MeteoSwiss/nwp-fdb-polytope-demo/main/examples/snapshots/feature_polygon_country_cut-out.html)

### ICON and native-grid notes

- [earthkit-data 0.19.0 release](https://github.com/ecmwf/earthkit-data/releases/tag/0.19.0)
- [MeteoSwiss native-grid branch example](https://github.com/MeteoSwiss/nwp-fdb-polytope-demo/blob/native_grid-16022026/examples/Polytope/work_with_data_native_grid.ipynb)

### Suggested next notebook

A natural follow-up is a small MeteoSwiss-specific exercise that reads one ICON GRIB, validates definitions, selects a parameter/time/level, plots on the native grid, and compares one validated regridding method with the native representation.
